# Bronze Layer, Customer Data Ingestion
**Source:** SQL Server (`customer_db.dbo.customer_data`)  
**Target:** HDFS Delta Table `/delta/bronze/customer`  
**Pattern:** Full Load with Ingestion Timestamp  
**Layer:** Bronze (Raw Ingestion)

---

In [1]:
import os, sys, logging
from pyspark.sql import SparkSession
from pyspark.sql.functions import current_timestamp

os.environ["SPARK_HOME"] = "/opt/spark"
os.environ["PYTHONPATH"] = "/opt/spark/python:/opt/spark/python/lib/py4j-0.10.9.7-src.zip"
sys.path.insert(0, "/opt/spark/python")
sys.path.insert(0, "/opt/spark/python/lib/py4j-0.10.9.7-src.zip")

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger(__name__)

CONFIG = {
    "sql_host"    : "host.docker.internal",
    "sql_port"    : "1433",
    "sql_db"      : "customer_db",
    "sql_user"    : "spark_user",
    "sql_pass"    : "SparkPass123!",
    "sql_table"   : "dbo.customer_data",
    "bronze_path" : "hdfs://master:8020/delta/bronze/customer",
    "database"    : "bronze",
    "table_name"  : "bronze_customer",
}

def get_spark():
    return (
        SparkSession.builder
        .appName("Bronze_Customer_Delta_Packages")
        .master("spark://master:7077")
        .config("spark.jars.packages", 
                "com.microsoft.sqlserver:mssql-jdbc:12.4.2.jre8,"  
                "io.delta:delta-spark_2.12:3.2.1")
        # Delta extensions
        .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
        .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
        # HDFS and Hive configs
        .config("spark.hadoop.fs.defaultFS", "hdfs://master:8020")
        .config("spark.sql.warehouse.dir", "hdfs://master:8020/user/hive/warehouse")
        .config("spark.hadoop.hive.metastore.uris", "thrift://master:9083")
        .config("spark.databricks.delta.stats.collect", "false")
        .enableHiveSupport()
        .getOrCreate()
    )

def extract(spark, cfg):
    jdbc_url = (f"jdbc:sqlserver://{cfg['sql_host']}:{cfg['sql_port']};"
                f"databaseName={cfg['sql_db']};encrypt=false;trustServerCertificate=true;")
    return spark.read.format("jdbc") \
        .option("url", jdbc_url) \
        .option("dbtable", cfg["sql_table"]) \
        .option("user", cfg["sql_user"]) \
        .option("password", cfg["sql_pass"]) \
        .option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver") \
        .load()

def run():
    logger.info("Pipeline Execution Started.")
    spark = get_spark()
    spark.sparkContext.setLogLevel("ERROR")

    df_raw = extract(spark, CONFIG)
    logger.info(f"Extracted {df_raw.count()} rows.")

    df_bronze = df_raw.withColumn("ingestion_timestamp", current_timestamp())

    spark.sql(f"CREATE DATABASE IF NOT EXISTS {CONFIG['database']}")
    spark.sql(f"DROP TABLE IF EXISTS {CONFIG['database']}.{CONFIG['table_name']}")

    logger.info(f"Writing Delta to {CONFIG['bronze_path']}...")
    df_bronze.write.format("delta") \
        .mode("overwrite") \
        .option("delta.columnMapping.mode", "name") \
        .save(CONFIG["bronze_path"])

    spark.sql(f"""
        CREATE TABLE {CONFIG['database']}.{CONFIG['table_name']}
        USING DELTA
        LOCATION '{CONFIG['bronze_path']}'
    """)

    count = spark.read.format("delta").load(CONFIG["bronze_path"]).count()
    logger.info(f"SUCCESS: {count} rows in Delta table.")
    spark.read.format("delta").load(CONFIG["bronze_path"]).show(5, truncate=False)

if __name__ == "__main__":
    run()

2026-03-13 14:39:07,396 [INFO] Pipeline Execution Started.


:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/jupyter/.ivy2/cache
The jars for the packages stored in: /home/jupyter/.ivy2/jars
com.microsoft.sqlserver#mssql-jdbc added as a dependency
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-355f0d90-c5e9-4eab-b134-c5f5d444a78a;1.0
	confs: [default]
	found com.microsoft.sqlserver#mssql-jdbc;12.4.2.jre8 in central
	found io.delta#delta-spark_2.12;3.2.1 in central
	found io.delta#delta-storage;3.2.1 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 179ms :: artifacts dl 9ms
	:: modules in use:
	com.microsoft.sqlserver#mssql-jdbc;12.4.2.jre8 from central in [default]
	io.delta#delta-spark_2.12;3.2.1 from central in [default]
	io.delta#delta-storage;3.2.1 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules         

+-----------+----------+---------------------+---------+-------------+-----------------+---+------+---------------+-----------------------+
|customer_id|name      |email                |country  |customer_type|registration_date|age|gender|total_purchases|ingestion_timestamp    |
+-----------+----------+---------------------+---------+-------------+-----------------+---+------+---------------+-----------------------+
|1          |Customer 1|customer1@example.com|Australia|Regular      |2011-05-15       |22 |Male  |191            |2026-03-13 14:39:22.985|
|2          |Customer 2|customer2@example.com|France   |Premium      |2018-11-27       |52 |Other |145            |2026-03-13 14:39:22.985|
|3          |Customer 3|customer3@example.com|Canada   |Premium      |2015-10-01       |32 |Other |691            |2026-03-13 14:39:22.985|
|4          |Customer 4|customer4@example.com|USA      |Premium      |2011-01-19       |70 |Other |644            |2026-03-13 14:39:22.985|
|5          |Custome